# Probability of Fire (PoF) — Forecasting of Subsection of Dataset

**Master:** Physics of Data \
**Course:** Laboratory of Computational Physics (LCP), Module B \
**Authors:** Gabriela Landinez Rangel, Andres Rojas Lozano, Fatemeh Dashti, Arash Taraz Jamshidi

*This notebook was created by us to present the final project for LCP MOD B, However it is based on public available code from the Probability of Fire project by ECMWF: https://ecmwf.github.io/AI-Probability-of-Fire/forecasting/. With this work, we aim to provide a clear, but friendly, guide with straightforward instructions and detailed explanations for students who, like us, want to reproduce this framework.*

In [2]:
import xarray as xr
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
import cftime
from xarray.coding.times import CFDatetimeCoder
import os
import cdsapi
import zipfile
import dask
from dask import delayed, compute
from dask.diagnostics import ProgressBar


## Select which reduced-feature for forecasting
| Model              | Features                     | Scientific question                  |
| ------------------ | ---------------------------- | ------------------------------------ |
| Full model         | all 13 features              | baseline                             |
| Weather-only       | `PR, T2, RH, WS`             | can weather alone explain fire risk? |
| Weather + moisture | `PR, T2, RH, WS, DF, DW, LF` | does fuel dryness improve the model? |
| Weather + human    | `PR, T2, RH, WS, PO, RD`     | do ignition proxies help?            |
| Moisture-only      | `DF, DW, LF`                 | how much comes from fuel moisture?   |


Only **one** `model_tag` should be active at a time. The other lines should remain commented with `#`.

In [ ]:
#model_tag = "weather_only" 

#model_tag = "weather_moisture"                

#model_tag = "weather_human"

#model_tag = "moisture_only"

## Forecasting 

In [3]:
# Folder structure
base_path = Path("./data/")
os.makedirs("./outputs/", exist_ok=True)

# Dates of Forecast
year = 2019
years=[year]
months = [12]
cds_months=[f'{m:02d}' for m in months]
cds_days= [f"{d:02d}" for d in range(1, 32)]

# LOAD WEATHER-ONLY MODEL

import joblib
import json


model = joblib.load(f"./data/POF_model_{model_tag}.joblib")

with open(f"./data/POF_features_{model_tag}.json", "r") as f:
    selected_features = json.load(f)

print("Loaded model:", model_tag)
print("Selected features:", selected_features)


Loaded model: full
Selected features: ['PR', 'T2', 'RH', 'WS', 'FU_LL', 'FU_LW', 'FU_DF', 'FU_DW', 'DF', 'DW', 'LF', 'PO', 'RD']


In [4]:

# Where we will store our predictions ---------------------------------------------
month = months[0]   # because months = [12]

output_file = f"./outputs/POF_prediction_{year}_{month:02d}_{model_tag}.nc"

print("Output file will be saved as:")
print(output_file)

Output file will be saved as:
./outputs/POF_prediction_2019_12_full.nc


In [5]:

cds_kwargs = {
    'url': 'https://cds.climate.copernicus.eu/api',
    'key': '7d185cca-7153-44d6-b716-0a154a2e96b1',
}

xds_kwargs = {
    'url': 'https://xds-preprod.ecmwf.int/api',
    'key': 'e26e7334-f401-4c4b-b8cf-8dc193987dfd',
}

In [6]:
for year in years:
    for month in cds_months: 
        for shortname,var in [["T2M","2m_temperature"],
                      ["D2M","2m_dewpoint_temperature"],
                      ["10U","10m_u_component_of_wind"],
                      ["10V","10m_v_component_of_wind"],
                      ["P","total_precipitation"]]:

            dataset = "reanalysis-era5-land"
            request = {
                "variable": [
                    var,
                ],
                "year": year,
                "month": month,
                "day": cds_days,
                "time": [
                    "00:00"
                ],
                "data_format": "netcdf",
                "download_format": "unarchived"
            }
            client = cdsapi.Client(**cds_kwargs)
            if not os.path.exists(f"data/{shortname}_{year}_{month}.nc"):
                client.retrieve(dataset, request, target=f"data/{shortname}_{year}_{month}.nc")
        if not os.path.exists(f"data/WS_{year}_{month}.nc"):
            with xr.open_dataset(f"data/10U_{year}_{month}.nc") as dsu,\
                xr.open_dataset(f"data/10V_{year}_{month}.nc") as dsv:
                dsw=dsu.u10**2+ dsv.v10**2
                dsw=dsw.rename("ws")
                dsw.to_netcdf(f"data/WS_{year}_{month}.nc")
                del dsw

        if not os.path.exists(f"data/RH_{year}_{month}.nc"):
            with xr.open_dataset(f"data/T2M_{year}_{month}.nc") as dst,\
                xr.open_dataset(f"data/D2M_{year}_{month}.nc") as dsd:
                es = 10**(7.5*(dst.t2m-273.15)/(237.7+(dst.t2m-273.15))) * 6.11
                e =  10**(7.5*(dsd.d2m-273.15)/(237.7+(dsd.d2m-273.15))) * 6.11
                dsrh = (e/es)*100
                dsrh = dsrh.rename("rh")
                dsrh.to_netcdf(f"data/RH_{year}_{month}.nc")
                del dsrh

In [7]:


# Open lat-lon reference dataset with chunks for Dask
lat_lon_data = xr.open_dataset(
    f"data/T2M_{years[0]}_{cds_months[0]}.nc",
    chunks={'latitude': 100, 'longitude': 100}  # adjust chunk size to your RAM
).sortby("latitude").sortby("longitude")

lat_lon_data = lat_lon_data.rename(valid_time="time", latitude='lat', longitude='lon')
lat_lon_data = lat_lon_data.drop_vars(['expver', 'number'])

# Function to download and regrid a single variable/month/year
@delayed
def process_variable(year, month, shortname, var):
    dataset = "derived-fire-fuel-biomass"
    request = {
        "variable": [var],
        "version": ["1"],
        "year": [f"{year}"],
        "month": [month]
    }

    client = cdsapi.Client(**xds_kwargs)
    zip_path = f"data/{shortname}_{year}_{month}.zip"
    nc_path = f"data/{shortname}_{year}_{month}.nc"
    regrid_path = f"data/{shortname}_{year}_{month}_R.nc"

    if os.path.exists(regrid_path):
        return regrid_path  # Already processed

    # Retrieve and extract
    client.retrieve(dataset, request, target=zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("data/")

    print(f"Regridding {shortname} for {year}-{month}")

    # Open with Dask chunks and interpolate
    with xr.open_dataset(nc_path, chunks={'latitude': 100, 'longitude': 100}) as ds_temp:
        ds_regrid = ds_temp.interp_like(lat_lon_data, method="linear")
        ds_regrid.to_netcdf(regrid_path)
        del ds_regrid

    return regrid_path

# Build delayed tasks for all combinations
tasks = []
for year in years:
    for month in cds_months:
        for shortname, var in [
            ["DFMC_MAP", "dead_fuel_moisture_content_group"],
            ["LFMC_MAP", "live_fuel_moisture_content_group"],
            ["FUEL_MAP", "fuel_group"]
        ]:
            tasks.append(process_variable(year, month, shortname, var))

# Execute in parallel using Dask (all cores)
results = compute(*tasks, scheduler='threads')  # use 'processes' if memory allows

C:\Users\sisto\AppData\Local\Temp\ipykernel_13492\3167100325.py:2: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 100. This could degrade performance. Instead, consider rechunking after loading.
  lat_lon_data = xr.open_dataset(
C:\Users\sisto\AppData\Local\Temp\ipykernel_13492\3167100325.py:2: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 100. This could degrade performance. Instead, consider rechunking after loading.
  lat_lon_data = xr.open_dataset(


In [8]:
# Open the Climate files
time_coder = CFDatetimeCoder(use_cftime=True)
PO = xr.open_dataset(base_path / "CLIMATE/POP_2020.nc")
RD = xr.open_dataset(base_path / "CLIMATE/road_density_2015_agg_r.nc")

# Extract the arrays
PO_arr = PO.population_density.values  
RD_arr = RD.road_length.values    

In [9]:
print("PO_arr:", PO_arr.shape)
print("RD_arr:", RD_arr.shape)

PO_arr: (1801, 3600)
RD_arr: (1801, 3600)


In [10]:
for year in years:
    for month in months:
        # Define the paths of the files
        ds_paths = {
            "AF": f"ACTIVE_FIRE_MAP_{year}_{month:02d}_R.nc",
            "FU": f"FUEL_MAP_{year}_{month:02d}_R.nc",
            "DF": f"DFMC_MAP_{year}_{month:02d}_R.nc",
            "LF": f"LFMC_MAP_{year}_{month:02d}_R.nc",
            "PR": f"P_{year}_{month:02d}.nc",
            "T2": f"T2M_{year}_{month:02d}.nc",
            "RH": f"RH_{year}_{month:02d}.nc",
            "WS": f"WS_{year}_{month:02d}.nc",
        }
        # Open all dynamic datasets
        with xr.open_dataset(base_path / ds_paths["FU"]) as FU, \
            xr.open_dataset(base_path / ds_paths["DF"]) as DF, \
            xr.open_dataset(base_path / ds_paths["LF"]) as LF, \
            xr.open_dataset(base_path / ds_paths["PR"]) as PR, \
            xr.open_dataset(base_path / ds_paths["T2"]) as T2, \
            xr.open_dataset(base_path / ds_paths["RH"]) as RH, \
            xr.open_dataset(base_path / ds_paths["WS"]) as WS, \
            xr.open_dataset(base_path / ds_paths["AF"]) as AF:

            n_days =len(AF.ACTIVE_FIRE)
            all_grids = []

            for i in range(n_days):
                # Extract arrays for timestep i
                FU_LL = FU.Live_Leaf[i].values
                FU_LW = FU.Live_Wood[i].values
                FU_DF = FU.Dead_Foliage[i].values
                FU_DW = FU.Dead_Wood[i].values
                DF_ = DF.DFMC_Foliage[i].values
                DW_ = DF.DFMC_Wood[i].values
                LF_ = LF.LFMC[i].values
                PR_ = PR.tp[i].values
                T2_ = T2.t2m[i].values
                RH_ = RH.rh[i].values
                WS_ = WS.ws[i].values

                # Mask where total fuel > 0
                ft = FU_LL + FU_LW + FU_DF + FU_DW
                mask = ft > 0

                # Flatten and build dataframe for prediction
                feature_arrays = {
                    "PR": PR_[mask],
                    "T2": T2_[mask],
                    "RH": RH_[mask],
                    "WS": WS_[mask],
                    "FU_LL": FU_LL[mask],
                    "FU_LW": FU_LW[mask],
                    "FU_DF": FU_DF[mask],
                    "FU_DW": FU_DW[mask],
                    "DF": DF_[mask],
                    "DW": DW_[mask],
                    "LF": LF_[mask],
                    "PO": PO_arr[mask],
                    "RD": RD_arr[mask],
                }
                X_pred = pd.DataFrame(feature_arrays)
                X_pred = X_pred[selected_features]
                print("Prediction features used:", list(X_pred.columns))
                print("Prediction matrix shape:", X_pred.shape)

                # Predict probability
                y_proba = model.predict_proba(X_pred)[:, 1]

                # Create full grid and fill masked values
                fire_prob_grid = np.full(ft.shape, np.nan, dtype=float)
                fire_prob_grid[mask] = y_proba
                all_grids.append(fire_prob_grid)

            # Stack all timesteps into 3D array (time, lat, lon)
            fire_prob_array = np.stack(all_grids, axis=0)
            
            # -----------------------------
            # SAVE TO NETCDF
            # -----------------------------
            time = [cftime.DatetimeJulian(year, month, day+1) for day in range(n_days)]

            ds_out = xr.Dataset(
                {"fire_probability": (["time", "latitude", "longitude"], fire_prob_array)},
                coords={
                    "time": time,
                    "latitude": PO.latitude,
                    "longitude": PO.longitude,
                }
            )
            os.remove(output_file) if os.path.exists(output_file) else None
            ds_out.to_netcdf(output_file)
            print(f"✅ Prediction saved → {output_file}")



Prediction features used: ['PR', 'T2', 'RH', 'WS', 'FU_LL', 'FU_LW', 'FU_DF', 'FU_DW', 'DF', 'DW', 'LF', 'PO', 'RD']
Prediction matrix shape: (3215547, 13)
Prediction features used: ['PR', 'T2', 'RH', 'WS', 'FU_LL', 'FU_LW', 'FU_DF', 'FU_DW', 'DF', 'DW', 'LF', 'PO', 'RD']
Prediction matrix shape: (3216253, 13)
Prediction features used: ['PR', 'T2', 'RH', 'WS', 'FU_LL', 'FU_LW', 'FU_DF', 'FU_DW', 'DF', 'DW', 'LF', 'PO', 'RD']
Prediction matrix shape: (0, 13)
Prediction features used: ['PR', 'T2', 'RH', 'WS', 'FU_LL', 'FU_LW', 'FU_DF', 'FU_DW', 'DF', 'DW', 'LF', 'PO', 'RD']
Prediction matrix shape: (0, 13)
Prediction features used: ['PR', 'T2', 'RH', 'WS', 'FU_LL', 'FU_LW', 'FU_DF', 'FU_DW', 'DF', 'DW', 'LF', 'PO', 'RD']
Prediction matrix shape: (0, 13)
Prediction features used: ['PR', 'T2', 'RH', 'WS', 'FU_LL', 'FU_LW', 'FU_DF', 'FU_DW', 'DF', 'DW', 'LF', 'PO', 'RD']
Prediction matrix shape: (0, 13)
Prediction features used: ['PR', 'T2', 'RH', 'WS', 'FU_LL', 'FU_LW', 'FU_DF', 'FU_DW', '

In [11]:
from pathlib import Path

print("Prediction files in outputs/")
for f in sorted(Path("outputs").glob("POF_prediction_*.nc")):
    print(f.name, round(f.stat().st_size / 1024 / 1024, 2), "MB")

Prediction files in outputs/
POF_prediction_2019_12.nc 1533.5 MB
POF_prediction_2019_12_full.nc 1533.5 MB
POF_prediction_2019_12_moisture_only.nc 1533.5 MB
POF_prediction_2019_12_weather_human.nc 1533.5 MB
POF_prediction_2019_12_weather_moisture.nc 1533.5 MB
POF_prediction_2019_12_weather_only.nc 1533.5 MB
